#Imports

In [ ]:
%load_ext cuml.accel

In [ ]:
# download TA-Lib
!wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
#!ls
!tar xvzf ta-lib-0.4.0-src.tar.gz
#!ls
import os
os.chdir('ta-lib') # Can't use !cd in co-lab
!./configure --prefix=/usr
!make
!make install
# wait ~ 30s
os.chdir('../')
#!ls
!pip install -q TA-Lib

import talib as ta
import time
import datetime as dt
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import numpy as np
import yfinance as yf
import pandas as pd
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning) #ignores "Dataframe is highly fragmented" warning.
from sklearn import preprocessing
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, mean_absolute_error, mean_absolute_percentage_error, r2_score, precision_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier,
                              StackingClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier, BaggingClassifier, VotingClassifier, IsolationForest)

ta-lib/
ta-lib/config.sub
ta-lib/aclocal.m4
ta-lib/CHANGELOG.TXT
ta-lib/include/
ta-lib/include/ta_abstract.h
ta-lib/include/ta_func.h
ta-lib/include/ta_common.h
ta-lib/include/ta_config.h.in
ta-lib/include/Makefile.am
ta-lib/include/ta_libc.h
ta-lib/include/ta_defs.h
ta-lib/missing
ta-lib/ta-lib.spec.in
ta-lib/config.guess
ta-lib/Makefile.in
ta-lib/ta-lib.dpkg.in
ta-lib/Makefile.am
ta-lib/autogen.sh
ta-lib/install-sh
ta-lib/configure
ta-lib/depcomp
ta-lib/HISTORY.TXT
ta-lib/configure.in
ta-lib/autom4te.cache/
ta-lib/autom4te.cache/output.0
ta-lib/autom4te.cache/requests
ta-lib/autom4te.cache/output.1
ta-lib/autom4te.cache/traces.0
ta-lib/autom4te.cache/traces.1
ta-lib/ltmain.sh
ta-lib/ta-lib-config.in
ta-lib/src/
ta-lib/src/ta_func/
ta-lib/src/ta_func/ta_MACDFIX.c
ta-lib/src/ta_func/ta_CDLPIERCING.c
ta-lib/src/ta_func/ta_DIV.c
ta-lib/src/ta_func/ta_ROCR100.c
ta-lib/src/ta_func/ta_ADXR.c
ta-lib/src/ta_func/ta_MAVP.c
ta-lib/src/ta_func/ta_CDLCLOSINGMARUBOZU.c
ta-lib/src/ta_func/ta_COSH.

KeyboardInterrupt: 

#Algorithms

In [ ]:
RF = RandomForestClassifier(random_state = 1,class_weight='balanced')
GB = GradientBoostingClassifier(random_state=0, learning_rate=0.05, )
ET = ExtraTreesClassifier(random_state = 1,class_weight='balanced')

clf = VotingClassifier(estimators=[('RF', RF),('GB', GB), ('ET', ET)], voting='soft')
clf

#Functions

In [ ]:
def change_days(date_str, days):
    import datetime as dt
    date = dt.datetime.strptime(date_str, "%Y-%m-%d")

    new_date = date + timedelta(days=days)

    return new_date.strftime("%Y-%m-%d")

def change_months(date_str, monthss):
  import datetime as dt
  date_obj = datetime.strptime(date_str, "%Y-%m-%d")

  new_date = date_obj + relativedelta(months=monthss)

  return new_date.strftime("%Y-%m-%d")

def simple_rsi(prices, timeperiod=14):
    # Calculate price changes (delta)
    delta = prices.diff()

    # Separate gains and losses
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    # Calculate RS using Simple Moving Average (SMA)
    avg_gain = gain.rolling(window=timeperiod).mean()
    avg_loss = loss.rolling(window=timeperiod).mean()

    rs = avg_gain / avg_loss

    # Apply your formula
    rsi = 100 - (100 / (1 + rs))
    return rsi

def features(stock_data, pred):
  stock_data = stock_data.rename(columns={'Close': 'Close-0d', 'High': 'High-0d', 'Low': 'Low-0d', 'Volume': 'Volume-0d'})

  '''
  for i in range(1, 5):
          stock_data[f"Close-{i}d"] = stock_data["Close-0d"].shift(i)
          stock_data[f"High-{i}d"] = stock_data["High-0d"].shift(i)
          stock_data[f"Low-{i}d"] = stock_data["Low-0d"].shift(i)
          stock_data[f"Volume-{i}d"] = stock_data["Volume-0d"].shift(i)
  '''

  if pred == 'RSI':
    stock_data["pred"] = simple_rsi(stock_data["Close-0d"], timeperiod=5)
  elif pred == 'ROC':
    stock_data['pred'] = ta.ROC(stock_data['Close-0d'], timeperiod=5)




  stock_data[f"CCI"] = ta.CCI(stock_data[f"High-0d"], stock_data[f"Low-0d"], stock_data[f"Close-0d"], timeperiod=5)
  stock_data['OBV'] = ta.OBV(stock_data['Close-0d'], stock_data['Volume-0d'])
  stock_data['OB-pred'] = ta.OBV(stock_data['Close-0d'], stock_data['pred']) # <----- [ =========================== EXPERIMENTAL =========================== ]
  stock_data['stoch-pred %K'], stock_data['stoch-pred %D'] = ta.STOCHRSI(stock_data['Close-0d'], timeperiod = 14, fastk_period=2, fastd_period=3, fastd_matype=0) # <----- [ =========================== EXPERIMENTAL =========================== ]
  stock_data['pred-PPO'] = ta.PPO(stock_data['pred'], fastperiod=3, slowperiod=5)




  stock_data[f'stochastic oscillator %K'], stock_data[f'stochastic oscillator %D'] = ta.STOCH(
      stock_data[f'High-0d'], stock_data[f'Low-0d'], stock_data[f'Close-0d'],
      fastk_period=2, slowk_period=5, slowk_matype=0, slowd_period=2, slowd_matype=0
  )




  time_periods = [2,3,4,5,6,7]


  for tp in time_periods:
      stock_data[f'OBV_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['OBV'], timeperiod=tp)
      stock_data[f'pred_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['pred'], timeperiod=tp)
      stock_data[f'CCI_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['CCI'], timeperiod=tp)
      stock_data[f'stoch_%K_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['stochastic oscillator %K'], timeperiod=tp)
      stock_data[f'Close_0d_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['Close-0d'], timeperiod=tp)
      stock_data[f'OB-pred_slope(tp={tp})'] = ta.LINEARREG_SLOPE(stock_data['OB-pred'], timeperiod=tp) # <----- [ =========================== EXPERIMENTAL =========================== ]


  for tp in time_periods:
      stock_data[f'OBV_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'OBV_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'pred_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'pred_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'CCI_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'CCI_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'stoch_%K_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'stoch_%K_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'Close_0d_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'Close_0d_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'OB-pred_slope(tp={tp})s_slope'] = ta.LINEARREG_SLOPE(stock_data[f'OB-pred_slope(tp={tp})'], timeperiod=tp)# <----- [ =========================== EXPERIMENTAL =========================== ]


      stock_data[f'OBV_slope(tp={tp}s_ROC'] = ta.ROC(stock_data[f'OBV_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'pred_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'pred_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'CCI_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'CCI_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'stoch_%K_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'stoch_%K_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'Close_0d_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'Close_0d_slope(tp={tp})'], timeperiod=tp)
      stock_data[f'OB-pred_slope(tp={tp})s_ROC'] = ta.ROC(stock_data[f'OB-pred_slope(tp={tp})'], timeperiod=tp)# <----- [ =========================== EXPERIMENTAL =========================== ]

  '''
  for i in range(1, 60):
    stock_data = stock_data.drop(columns=[f"Close-{i}d", f"High-{i}d", f"Low-{i}d", f'Volume-{i}d'])
  '''

  return stock_data

def Training_data(pred, stock, startdate, enddate, shuffle=False, clean = True, classification_type = 'change'):
  stock_data = yf.Ticker(stock).history(start = startdate, end=enddate)
  stock_data = features(stock_data, pred)

  stock_data["future"] = stock_data['pred'].shift(-5)
  stock_data["future_change"] = (stock_data["future"]-stock_data[f"pred"])

  if classification_type.lower() == 'change':
    stock_data["label"] = np.where(stock_data["future_change"] > 0, 1,0)
  elif classification_type.lower() == 'value':
    stock_data["label"] = np.where(stock_data['future'] > 50, 1, 0)
  else:
    print('Please Enter A Suitable Classification Type.')
    return 'Please Enter A Suitable Classification Type.', 'Please Enter A Suitable Classification Type.'


  # Handle infinite and Nan values
  stock_data[np.isinf(stock_data)] = np.nan
  stock_data = stock_data.dropna()

  stock_data = stock_data.drop(columns=['future','future_change'])
  stock_data = stock_data.drop(columns=[ 'Stock Splits', 'Dividends'])

  if shuffle:
    stock_data = stock_data.sample(frac=1, random_state = 1)

  label = pd.DataFrame()
  label["label"] = stock_data.pop("label")

  if clean:
    iso = IsolationForest(random_state=0)
    iso.fit(stock_data)
    outliers = iso.predict(stock_data)
    stock_data = stock_data[outliers == 1]
    label = label[outliers==1]

  return stock_data, label

def Pred_data(pred, stock, date):
  data = yf.Ticker(stock).history(start = change_months(date, -10), end = date)
  data = features(data, pred)
  data = data.drop(columns = ['Stock Splits', 'Dividends'])
  return data.tail(1)

def model(stock, pred, date, clf):
  if pred == 'RSI':
    X,y = Training_data('RSI', stock, change_months(date, -100), date)
  elif pred == 'ROC':
    X,y = Training_data('ROC', stock, change_months(date, -100), date)
  clf.fit(X,y.values.ravel())

  pred_data = Pred_data(pred, stock, date)
  y_pred = clf.predict(pred_data)
  y_proba = clf.predict_proba(pred_data)

  return y_pred, y_proba

def backtest(stock, date, stoploss = 0.1 , traceback = -5,  action = 'long'):#function will return how much the stock price changed in the coming month with exit rules
  enddate = change_days(date, abs(traceback)*4)
  print('\n',f"[------=====◼️◼️◼️◼️◼️ Backtesting {stock} {action} {date}◼️◼️◼️◼️◼️=====--------]")

  ticker = yf.Ticker(stock).history(start = change_days(date, -7), end = enddate)
  ticker['Future'] = ticker['Close'].shift(traceback)

  while True:
    try:
      prices = ticker.loc[f'{date}']
      break
    except:
      date = change_days(date, -1)
      pass

  initial_price = prices['Close']
  new_price = prices['Future']

  #print(order)
  pct_change = ((new_price-initial_price)/initial_price)

  print(f' initial close: {initial_price}, new close: {new_price}')

  if action == 'long':
    order = yf.Ticker(stock).history(start=date, end=enddate)['Low']
    stop_loss_price = float(initial_price)*(1-stoploss)
    for i in order:
      if i < stop_loss_price:
        print(f"[---------🔴 Long {stock} price decreased by over {stoploss*100}%, stop loss triggered 🔴 ---------]")
        return -stoploss

    print(f"[--------- {stock} Percentage Gain: {pct_change} ---------]")
    return pct_change

  elif action =='short':
    order = yf.Ticker(stock).history(start=date, end=enddate)['High']
    stop_loss_price = float(initial_price)*(1+stoploss)
    for i in order:
      if i > stop_loss_price:

        print(f"[---------🔴 Short {stock} price increased by over {stoploss*100}%, stop loss triggered 🔴---------]")
        return -stoploss

    print(f"[--------- {stock} Percentage Gain: {pct_change*-1} ---------]")
    return pct_change*-1

  else:
    print('🟥🟥🟥🟥🟥 PLEASE ENTER ACTION (LONG/SHORT) 🟥🟥🟥🟥🟥')
  return pct_change

#Model

In [ ]:
# Complete S&P500 stock list
stocks= ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COIN', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DAY', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EMR', 'ENPH', 'ETR', 'EOG', 'EPAM', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'ES', 'EXC', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS', 'FICO', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FSLR', 'FE', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEHC', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC', 'ICE', 'IFF', 'IP', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LW', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LKQ', 'LMT', 'L', 'LOW', 'LULU', 'LYB', 'MTB', 'MPC', 'MKTX', 'MAR', 'MMC', 'MLM', 'MAS', 'MA', 'MTCH', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'MHK', 'MOH', 'TAP', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PH', 'PAYX', 'PAYC', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'POOL', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VTR', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VTRS', 'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']
len(stocks)

493

In [ ]:
# S&P500 stock list for certain year
stocks= ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DAY', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EMR', 'ENPH', 'ETR', 'EOG', 'EPAM', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'ES', 'EXC', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS', 'FICO', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FSLR', 'FE', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC', 'ICE', 'IFF', 'IP', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LW', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LKQ', 'LMT', 'L', 'LOW', 'LULU', 'LYB', 'MTB', 'MPC', 'MKTX', 'MAR', 'MLM', 'MAS', 'MA', 'MTCH', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'MHK', 'MOH', 'TAP', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PH', 'PAYX', 'PAYC', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'POOL', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VTR', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VTRS', 'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']
len(stocks)

In [ ]:
def complete_model(date, stocks):
  df = pd.DataFrame()
  bullarity = []

  for i,stock in enumerate(stocks): # THIS TAKES AROUND 10 MINUTES, GIVE OR TAKE.
    try:
      RSI = Pred_data('RSI', stock, date)['pred'].iloc[0]
      ROC = Pred_data('ROC', stock, date)['pred'].iloc[0]
      if (RSI > 50) & (ROC > 0):
        bullarity.append('BULL')
      elif (RSI < 50) & (ROC < 0):
        bullarity.append('BEAR')
      else:
        bullarity.append([RSI, ROC])
    except Exception as e:
      print('🟥🟥🔴', stock, e, '\n')


  print(f'''
  Bulls: {bullarity.count('BULL')}
  Bears: {bullarity.count('BEAR')}
  ''')

  for i, stock in enumerate(stocks):
    try:
      RSI_pred, RSI_proba = model(stock, 'RSI', date, clf)
      ROC_pred, ROC_proba = model(stock, 'ROC', date, clf)

      RSI_bullproba = RSI_proba[0][1]
      ROC_bullproba = ROC_proba[0][1]

      harmonic_mean = (2*(RSI_bullproba*ROC_bullproba))/(RSI_bullproba + ROC_bullproba)
      pred = np.where(harmonic_mean>0.5,1,0)

      print(f'''[-------------------- {stock.upper()} {bullarity[i], pred} {i/len(stocks)} ----------------------]
      RSI models proba: {RSI_proba}
      ROC models proba: {ROC_proba}
      harmonic mean bull proba: {harmonic_mean}
      ''')
      df[f'{stock}'] = [harmonic_mean, bullarity[i]]
    except Exception as e:
      print('🟥🟥🔴', stock, e, '\n')

  return df

In [ ]:
model('aapl', 'RSI', '2020-01-01', clf)

(array([0]), array([[0.78235681, 0.21764319]]))

In [ ]:
date = '2021-01-01'

for i in range(10):
  df = complete_model(date, stocks)
  df.to_excel(f'{date}, RSI-ROC model predictions, (vote).xlsx')
  df.to_excel(f'/content/drive/MyDrive/SP/Testing/RSI-ROC model/{date}(vote), RSI-ROC model predictions.xlsx')
  date = change_days(date, 7)

Streaming output truncated to the last 5000 lines.
      ROC models proba: [[0.40050555 0.59949445]]
      harmonic mean bull proba: 0.6109227160178897
      
[-------------------- GWW ('BEAR', array(1)) 0.950920245398773 ----------------------]
      RSI models proba: [[0.46292037 0.53707963]]
      ROC models proba: [[0.40419229 0.59580771]]
      harmonic mean bull proba: 0.5649214568683807
      
[-------------------- WAB ('BULL', array(0)) 0.9529652351738241 ----------------------]
      RSI models proba: [[0.88301073 0.11698927]]
      ROC models proba: [[0.93214976 0.06785024]]
      harmonic mean bull proba: 0.08588802107214988
      
[-------------------- WMT ('BULL', array(0)) 0.9550102249488752 ----------------------]
      RSI models proba: [[0.84960586 0.15039414]]
      ROC models proba: [[0.72733896 0.27266104]]
      harmonic mean bull proba: 0.193859448617204
      
[-------------------- DIS ('BEAR', array(1)) 0.9570552147239264 ----------------------]
      RSI models

In [ ]:
df = df.T
stocks = list(df.iloc[0])
df.columns = stocks
df = df.drop('Unnamed: 0')
df

,ADBE,A,ALLE,MO,AEP,AON,APO,ACGL,ADM,BALL,...,REG,SBAC,TRGP,TGT,TSLA,TSN,VTRS,VICI,WEC,WELL
0,0.5712,0.544134,0.583958,0.544878,0.564433,0.61008,0.466044,0.514795,0.506005,0.578582,...,0.650913,0.566798,0.4738,0.644883,0.444893,0.574527,0.552489,0.613614,0.510801,0.492622
1,BULL,BULL,BULL,BULL,BULL,BULL,BEAR,BULL,BULL,BULL,...,BULL,BULL,BEAR,BULL,BEAR,BULL,BULL,BULL,BULL,BEAR


In [ ]:
df.to_excel("/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2025-02-01, RSI-ROC model predictions.xlsx")

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2025-02-01, RSI-ROC model predictions.xlsx")
df.shape

(62, 3)

In [ ]:
# Drop stocks which has RSI/ROC that are not aligned wtih prediction. These stocks' directions are rather useless (a small buffer could be added, in the previous code block when determining bull/bear)
drop_cols = []

for i in range(len(df.iloc[0])):
  if (df.iloc[0,i] > 0.5) & (df.iloc[1, i]=='BULL'):
    continue
  elif (df.iloc[0,i] < 0.5) & (df.iloc[1,i]=='BEAR'):
    continue
  else:
    drop_cols.append(df.columns[i])

df = df.drop(columns = drop_cols)

# Sorted the stocks in order of bull probability
df = df.T
df_bull = df[df.iloc[:, 1] == 'BULL']
df_bear = df[df.iloc[:, 1] == 'BEAR']
sorted_df_bull = df_bull.sort_values(by=[0], ascending=False)
sorted_df_bear = df_bear.sort_values(by=[0], ascending=True)

bull_count = len(list(sorted_df_bull.index))
bear_count = len(list(sorted_df_bear.index))

'(bull count, bear count)', bull_count, bear_count

('(bull count, bear count)', 18, 16)

In [ ]:
# Picks the bullest and bearest X stocks

stocks_bought = 10
k1 = round((bull_count/(bull_count+bear_count))*stocks_bought)
k2 = round((bear_count/(bull_count+bear_count))*stocks_bought)

bull_stocks = list(sorted_df_bull.index)[:k1]
bear_stocks = list(sorted_df_bear.index)[:k2]
print(f'''
Bull Stocks: x{len(bull_stocks)}
  {bull_stocks}

Bear_stocks: x{len(bear_stocks)}
  {bear_stocks}
''')


Bull Stocks: x5
  ['WBD', 'D', 'DPZ', 'YUM', 'ATO']

Bear_stocks: x5
  ['HII', 'UAL', 'CMCSA', 'HST', 'CL']



In [ ]:
changes = []
date = '2021-02-05'
stoploss = 0.04
for stock in bull_stocks:
  changes.append(backtest(stock, date, stoploss=stoploss, action='long')+1)
for stock in bear_stocks:
  changes.append((backtest(stock, date, stoploss=stoploss, action='short'))+1)

print('\n'*5, sum(changes)/len(changes))


 [------=====◼️◼️◼️◼️◼️ Backtesting WBD long 2021-02-05◼️◼️◼️◼️◼️=====--------]
 initial close: 42.66999816894531, new close: 47.79999923706055
[---------🔴 Long WBD price decreased by over 4.0%, stop loss triggered 🔴 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting D long 2021-02-05◼️◼️◼️◼️◼️=====--------]
 initial close: 58.565006256103516, new close: 57.0798454284668
[--------- D Percentage Gain: -0.025359184990814177 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting DPZ long 2021-02-05◼️◼️◼️◼️◼️=====--------]
 initial close: 352.3016662597656, new close: 359.7170104980469
[--------- DPZ Percentage Gain: 0.021048280347370343 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting YUM long 2021-02-05◼️◼️◼️◼️◼️=====--------]
 initial close: 95.52727508544922, new close: 96.10356903076172
[--------- YUM Percentage Gain: 0.006032768597209589 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting ATO long 2021-02-05◼️◼️◼️◼️◼️=====--------]
 initial close: 78.0816650390625, new close: 79.8353042602539
[---

#Auto read and test files


In [ ]:
#prints all file paths in a folder. I know i can just use this loop but it seems complicated
import os

folder_path = '/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies'

# List all files and folders in the path
for file_name in os.listdir(folder_path):
    # Print the full path
    print(f'"{os.path.join(folder_path, file_name)}",')

"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2025-02-01, RSI-ROC model predictions.xlsx",
"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2018-10-01, RSI-ROC model predictions.xlsx",
"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2022-08-01, RSI-ROC model predictions.xlsx",
"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-03-04 RSI-ROC model predictions(ET).xlsx",
"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2025-01-01, RSI-ROC model predictions (ET).xlsx",
"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-01, RSI-ROC model predictions, (RandomForestClassifier(class_weight='balanced', random_state=1)).xlsx",
"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-08, RSI-ROC model predictions, (RandomForestClassifier(class_weight='balanced', random_state=1)).xlsx",
"/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-15, RSI-ROC model predictions, (RandomForestClassifier(class_we

In [ ]:
files = ["/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2025-02-01, RSI-ROC model predictions.xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2018-10-01, RSI-ROC model predictions.xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2022-08-01, RSI-ROC model predictions.xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-03-04, RSI-ROC model predictions(ET).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2025-01-01, RSI-ROC model predictions (ET).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-01, RSI-ROC model predictions, (RandomForestClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-08, RSI-ROC model predictions, (RandomForestClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-15, RSI-ROC model predictions, (RandomForestClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-22, RSI-ROC model predictions, (RandomForestClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2020-01-29, RSI-ROC model predictions, (RandomForestClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2021-01-01, RSI-ROC model predictions, (ExtraTreesClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2021-01-08, RSI-ROC model predictions, (ExtraTreesClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2021-01-15, RSI-ROC model predictions, (ExtraTreesClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2021-01-22, RSI-ROC model predictions, (ExtraTreesClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2021-01-29, RSI-ROC model predictions, (ExtraTreesClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2021-02-05, RSI-ROC model predictions, (ExtraTreesClassifier(class_weight='balanced', random_state=1)).xlsx",
        "/content/drive/MyDrive/SP/Testing/RSI-ROC model/no copies/2021-02-12, RSI-ROC model predictions, (ExtraTreesClassifier(class_weight='balanced', random_state=1)).xlsx",
]

In [ ]:
dates = []
for i, file_ in enumerate(files):
  date = files[i].split('/')[-1]
  date = date.split(',')[0]
  dates.append(date)

In [ ]:
# This code evaluates trading accuracy by running a backtest on stocks labeled 'BULL' or 'BEAR'.
# It records success (1) or failure (0) for each trade decision, then calculates overall accuracy.
#AT = 0.5ish, 685 samples

def accuracy_checker(stock_list, bullarity, date, traceback, AT):
  for stock in stock_list:
    if backtest(stock, date, stoploss = 1, traceback = traceback, action = 'long' if bullarity == 'BULL' else 'short') > 0:
      AT.append(1)
    else:
      AT.append(0)


Accuracy_tracker = AT = []

for i, file_ in enumerate(files):


  date = dates[i]
  df = pd.read_excel(file_)



  drop_cols = []

  for i in range(len(df.iloc[0])):
    if (df.iloc[0,i] > 0.5) & (df.iloc[1, i]=='BULL'):
      continue
    elif (df.iloc[0,i] < 0.5) & (df.iloc[1,i]=='BEAR'):
      continue
    else:
      drop_cols.append(df.columns[i])

  df = df.drop(columns = drop_cols)
  print(df.shape, '🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢')



  bull_stocks = list(df.columns[df.iloc[1, :] == 'BULL'])
  bear_stocks = list(df.columns[df.iloc[1, :] == 'BEAR'])

  accuracy_checker(bull_stocks, 'BULL', date, -5, AT)
  accuracy_checker(bear_stocks, 'Bear', date, -5, AT)

print(AT.count(1)/len(AT))

In [ ]:
len(AT)

[0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,


In [ ]:
backtest('aapl', '2020-01-01', stoploss = 1, traceback = -5, action = 'long')


 [------=====◼️◼️◼️◼️◼️ Backtesting aapl long 2020-01-01◼️◼️◼️◼️◼️=====--------]
 initial close: 70.78544616699219, new close: 73.08509826660156
[--------- aapl Percentage Gain: 0.03248764010308267 ---------]


np.float64(0.03248764010308267)

In [ ]:
weekly_changes = []
winrates = []

for j, file_ in enumerate(files):
  df = pd.read_excel(file_)
  print(df.shape)
  # Drop stocks which has RSI/ROC that are not aligned wtih prediction. These stocks' directions are rather useless (a small buffer could be added, in the previous code block when determining bull/bear)
  drop_cols = []

  for i in range(len(df.iloc[0])):
    if (df.iloc[0,i] > 0.5) & (df.iloc[1, i]=='BULL'):
      continue
    elif (df.iloc[0,i] < 0.5) & (df.iloc[1,i]=='BEAR'):
      continue
    else:
      drop_cols.append(df.columns[i])

  df = df.drop(columns = drop_cols)

  # Sorted the stocks in order of bull probability
  df = df.T
  df_bull = df[df.iloc[:, 1] == 'BULL']
  df_bear = df[df.iloc[:, 1] == 'BEAR']
  sorted_df_bull = df_bull.sort_values(by=[0], ascending=False)
  sorted_df_bear = df_bear.sort_values(by=[0], ascending=True)

  bull_count = len(list(sorted_df_bull.index))
  bear_count = len(list(sorted_df_bear.index))

  print('(bull count, bear count)', bull_count, bear_count)


  # Picks the bullest and bearest X stocks
  stocks_bought = 10
  k1 = round((bull_count/(bull_count+bear_count))*stocks_bought)
  k2 = round((bear_count/(bull_count+bear_count))*stocks_bought)

  bull_stocks = list(sorted_df_bull.index)[:k1]
  bear_stocks = list(sorted_df_bear.index)[:k2]
  print(f'''
  Bull Stocks: x{len(bull_stocks)}
    {bull_stocks}

  Bear_stocks: x{len(bear_stocks)}
    {bear_stocks}
  ''')

  changes = []
  date = dates[j]
  stoploss = 0.9999
  traceback = -5

  for stock in bull_stocks:
    changes.append(backtest(stock, date, stoploss=stoploss,traceback=traceback, action='long')+1)
  for stock in bear_stocks:
    changes.append((backtest(stock, date, stoploss=stoploss,traceback=traceback,  action='short'))+1)

  winrate = sum(1 for i in changes if i>1)/len(changes)

  print('\n'*5, sum(changes)/len(changes))
  weekly_changes.append(sum(changes)/len(changes))
  winrates.append(winrate)

(2, 63)
(bull count, bear count) 53 9

  Bull Stocks: x9
    ['PAYC', 'PLD', 'JKHY', 'REG', 'BLDR', 'EL', 'TGT', 'BAX', 'CSX']

  Bear_stocks: x1
    ['C']
  

 [------=====◼️◼️◼️◼️◼️ Backtesting PAYC long 2025-02-01◼️◼️◼️◼️◼️=====--------]
 initial close: 205.52392578125, new close: 197.82022094726562
[--------- PAYC Percentage Gain: -0.0374832506955119 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting PLD long 2025-02-01◼️◼️◼️◼️◼️=====--------]
 initial close: 114.2291259765625, new close: 112.40911865234375
[--------- PLD Percentage Gain: -0.015932953252151983 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting JKHY long 2025-02-01◼️◼️◼️◼️◼️=====--------]
 initial close: 171.15313720703125, new close: 170.8778839111328
[--------- JKHY Percentage Gain: -0.0016082281656659557 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting REG long 2025-02-01◼️◼️◼️◼️◼️=====--------]
 initial close: 68.32960510253906, new close: 70.403076171875
[--------- REG Percentage Gain: 0.03034513467807074 ---------]



In [ ]:
sum(weekly_changes)/len(weekly_changes), sum(winrates)/len(winrates)

(np.float64(1.0045526561486655), 0.5117647058823529)

#Other


In [ ]:
error = []
for stock in stocks:
  ticker = yf.Ticker(stock).history(start = '2019-01-01', end='2021-03-04')
  if ticker.empty:
    error.append(stock)

for i in error:
  stocks.remove(i)

print(stocks)

ERROR:yfinance:$COIN: possibly delisted; no price data found  (1d 2019-01-01 -> 2021-03-04) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1614834000")
ERROR:yfinance:$CEG: possibly delisted; no price data found  (1d 2019-01-01 -> 2021-03-04) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1614834000")
ERROR:yfinance:$GEHC: possibly delisted; no price data found  (1d 2019-01-01 -> 2021-03-04) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1614834000")


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DAY', 'D

In [ ]:
backtest('^spx', '2020-01-29', 1, 'long')


 [------=====◼️◼️◼️◼️◼️ Backtesting ^spx long 2020-01-29◼️◼️◼️◼️◼️=====--------]
 initial close: 3273.39990234375, new close: 3334.68994140625
[--------- ^spx Percentage Gain: 0.018723663741364572 ---------]


np.float64(0.018723663741364572)

In [ ]:
ticker = yf.Ticker('ABNB').history(start = '2018-01-01', end='2020-01-01')
print(ticker)

ERROR:yfinance:$ABNB: possibly delisted; no price data found  (1d 2018-01-01 -> 2020-01-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1577854800")


Empty DataFrame
Columns: [Open, High, Low, Close, Adj Close, Volume]
Index: []


In [ ]:
lists = data = [
    ['ROST', 'MNST', 'WDC', 'MSI', 'CHRW'],
    ['WDC', 'MNST', 'ROST', 'CHRW', 'MSI'],
    ['WDC', 'MNST', 'ROST', 'CHRW', 'CTAS'],
    ['WDC', 'ROST', 'MNST', 'CHRW', 'MSI'],
    ['MNST', 'CHRW', 'ROST', 'IRM', 'MSI'],
    ['IRM', 'CTAS', 'ROST', 'MNST', 'CHRW'],
    ['MNST', 'ROST', 'CHRW', 'IRM', 'DLR'],
    ['ROST', 'MNST', 'IRM', 'CHRW', 'IVZ'],
    ['MNST', 'ROST', 'IRM', 'MSI', 'CHRW'],
]

date = '2026-01-22'
stoploss = 1
traceback = -22
action = 'long'

winrate = []
avg_changes = []
info_storage_changes = []

for list_ in lists:
  changes = []

  for stock in list_:
    changes.append(backtest(stock, date, stoploss, traceback, action)+1)

  winrates.append(sum(1 for i in changes if i > 1)/len(changes))
  info_storage_changes.append(changes)
  avg_changes = sum(changes)/len(changes)

  date = change_days(date, 1)


 [------=====◼️◼️◼️◼️◼️ Backtesting ROST long 2026-01-22◼️◼️◼️◼️◼️=====--------]
 initial close: 186.6134033203125, new close: 200.12469482421875
[--------- ROST Percentage Gain: 0.07240257807588879 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting MNST long 2026-01-22◼️◼️◼️◼️◼️=====--------]
 initial close: 80.88999938964844, new close: 85.54000091552734
[--------- MNST Percentage Gain: 0.05748549339801294 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting WDC long 2026-01-22◼️◼️◼️◼️◼️=====--------]
 initial close: 243.17359924316406, new close: 270.4405822753906
[--------- WDC Percentage Gain: 0.11212970123849937 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting MSI long 2026-01-22◼️◼️◼️◼️◼️=====--------]
 initial close: 396.4425048828125, new close: 469.60919189453125
[--------- MSI Percentage Gain: 0.18455812913740582 ---------]

 [------=====◼️◼️◼️◼️◼️ Backtesting CHRW long 2026-01-22◼️◼️◼️◼️◼️=====--------]
 initial close: 177.29354858398438, new close: 176.7454376220703
[--------- CHR

In [ ]:
list_

['MNST', 'ROST', 'IRM', 'MSI', 'CHRW']

In [ ]:
avg_changes

(np.float64(1.087530531391858),
 [0.1,
  0.6,
  0.8,
  0.5,
  0.4,
  0.6,
  0.6,
  0.7,
  0.5,
  0.6,
  0.4,
  0.4,
  0.4,
  0.5,
  0.6,
  0.5,
  0.5,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.8,
  0.8,
  0.8,
  0.8,
  1.0,
  1.0,
  1.0,
  0.4,
  0.6])